# Fair full-data live semantic HNSW benchmark

This is the validation run after the live predicate result. It addresses the main remaining fairness concern: the previous `hnsw_rs` baselines used `DistCosine`, while the custom traversal used direct dot products over already-normalized embeddings.

This notebook patches the Rust live benchmark so **the HNSW library baseline and our custom traversal use the same normalized-dot kernel and the same graph**. It also defaults to the full ~44k fashion source dataset (strict held-out test split is the indexed search set).

The live method still executes the real Binary1-LS2-int4 programs inside timed traversal. We sweep semantic selectivity and compare recall, p50/p95 latency, predicate evaluations, and exact result parity.


In [ ]:
#@title 1) Settings
FULL_DATA = True #@param {type:"boolean"}
QUERIES = 100 #@param {type:"integer"}
K = 50 #@param {type:"integer"}
EF = 128 #@param {type:"integer"}
M = 24 #@param {type:"integer"}
EF_CONSTRUCTION = 200 #@param {type:"integer"}
POSTFILTER_OVERSAMPLE = 8 #@param {type:"integer"}
POSITIVE = 'minimalist,office_appropriate' #@param {type:"string"}
NEGATIVE = 'technical_sporty' #@param {type:"string"}
TARGET_FRACTIONS = '0.50,0.20,0.10,0.05,0.02' #@param {type:"string"}
print({'FULL_DATA':FULL_DATA,'QUERIES':QUERIES,'K':K,'EF':EF,'positive':POSITIVE,'negative':NEGATIVE,'fractions':TARGET_FRACTIONS})


In [ ]:
#@title 2) Clone repo + dependencies
import os, pathlib, shutil, subprocess, sys
ROOT = pathlib.Path('/content/ras')
if ROOT.exists(): shutil.rmtree(ROOT)
subprocess.run(['git','clone','--depth=1','https://github.com/hanialshater/ras.git',str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.','faiss-cpu'], check=True)
if shutil.which('rustc') is None or shutil.which('cargo') is None:
    print('Installing minimal Rust toolchain...')
    subprocess.run(['bash','-lc', "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal"], check=True)
    os.environ['PATH'] = str(pathlib.Path.home()/'.cargo'/'bin') + os.pathsep + os.environ.get('PATH','')
print('commit:', subprocess.check_output(['git','rev-parse','HEAD']).decode().strip())
print('python:', sys.version.split()[0])
print('rustc:', subprocess.check_output(['rustc','--version']).decode().strip())


In [ ]:
#@title 3) Export real fashion assets
import pathlib, shutil, subprocess, sys, time, os
os.chdir('/content/ras')
CFG = 'configs/binary_bbq.yaml' if FULL_DATA else 'configs/binary_bbq_smoke.yaml'
ASSETS = pathlib.Path('/content/semantic_hnsw_fair_assets')
if ASSETS.exists(): shutil.rmtree(ASSETS)
t0=time.time()
subprocess.run([sys.executable,'-m','experiments.export_native_finalists','--config',CFG,'--out-dir',str(ASSETS)], check=True)
print(f'export finished in {(time.time()-t0)/60:.1f} min')
print('assets:', ASSETS)
print('programs:', sorted(p.name for p in (ASSETS/'sidecar_programs').iterdir() if p.is_dir()))


In [ ]:
#@title 4) Verify normalization and patch HNSW to the exact normalized-dot distance
import numpy as np, pathlib
items = np.fromfile(ASSETS/'fp32_items.f32', dtype=np.float32).reshape(-1,384)
norms = np.linalg.norm(items, axis=1)
print('indexed items:', len(items))
print('norm min/mean/max:', float(norms.min()), float(norms.mean()), float(norms.max()))
print('max |norm-1|:', float(np.max(np.abs(norms-1))))
assert np.max(np.abs(norms-1)) < 2e-3, 'retrieval embeddings are not sufficiently unit-normalized for normalized-dot distance'

src = pathlib.Path('/content/ras/rust/semantic_engine/src/bin/semantic_hnsw_live.rs')
text = src.read_text()
old_import = 'use hnsw_rs::prelude::{DistCosine, Hnsw};'
assert old_import in text, 'unexpected semantic_hnsw_live.rs import; inspect current source'
text = text.replace(old_import, 'use hnsw_rs::prelude::{Distance, Hnsw};')
marker = 'const D: usize = 384;\n'
dist = r'''

#[derive(Clone, Copy, Default)]
struct DistNormalizedDot;

impl Distance<f32> for DistNormalizedDot {
    #[inline(always)]
    fn eval(&self, va: &[f32], vb: &[f32]) -> f32 {
        debug_assert_eq!(va.len(), vb.len());
        let mut s = 0.0f32;
        for i in 0..va.len() {
            s += unsafe { *va.get_unchecked(i) } * unsafe { *vb.get_unchecked(i) };
        }
        1.0 - s.clamp(-1.0, 1.0)
    }
}
'''
assert marker in text
text = text.replace(marker, marker + dist, 1)
text = text.replace('DistCosine', 'DistNormalizedDot')
src.write_text(text)
assert 'DistCosine' not in text
assert 'DistNormalizedDot' in text
print('Patched library HNSW and extracted-graph traversal to the same normalized-dot geometry.')


In [ ]:
#@title 5) Compile the fair live Rust benchmark
import subprocess, time, os
os.chdir('/content/ras')
t0=time.time()
subprocess.run(['cargo','build','--release','--manifest-path','rust/semantic_engine/Cargo.toml','--bin','semantic_hnsw_live'], check=True)
BIN='/content/ras/rust/semantic_engine/target/release/semantic_hnsw_live'
print(f'compiled in {time.time()-t0:.1f}s:', BIN)


In [ ]:
#@title 6) Convert requested selectivities into semantic gates
import numpy as np, pandas as pd, sys, pathlib, importlib
SRC = str(pathlib.Path('/content/ras/src'))
sys.path[:] = [p for p in sys.path if p != SRC]
sys.path.insert(0,SRC)
for name in list(sys.modules):
    if name == 'ras' or name.startswith('ras.'): del sys.modules[name]
importlib.invalidate_caches()
from ras import SemanticExecutor
pos=[x.strip() for x in POSITIVE.split(',') if x.strip()]
neg=[x.strip() for x in NEGATIVE.split(',') if x.strip()]
n_pred=len(pos)+len(neg)
executor=SemanticExecutor.open(str(ASSETS/'sidecar_index'),str(ASSETS/'sidecar_programs'))
ids=np.arange(executor.index.n_items,dtype=np.int64)
sem_mean=executor.score_candidates(ids,positive=pos,negative=neg)/max(1,n_pred)
rows=[]
for f in [float(x) for x in TARGET_FRACTIONS.split(',') if x.strip()]:
    if f*len(ids) < K+1:
        print('skip',f,'not enough eligible items'); continue
    gate=float(np.quantile(sem_mean,1-f))
    actual=float(np.mean(sem_mean>=gate))
    rows.append({'target_fraction':f,'gate_logprob':gate,'actual_fraction_python':actual,'eligible_items':int((sem_mean>=gate).sum())})
gate_df=pd.DataFrame(rows).sort_values('target_fraction',ascending=False).reset_index(drop=True)
display(gate_df)
assert len(gate_df)


In [ ]:
#@title 7) Run fair live-vs-baseline sweep
import subprocess, pathlib, time, pandas as pd
all_runs=[]
for r in gate_df.itertuples(index=False):
    frac=float(r.target_fraction); gate=float(r.gate_logprob)
    out=pathlib.Path(f'/content/semantic_hnsw_fair_{frac:.3f}.csv')
    cmd=[BIN,'--assets',str(ASSETS),'--programs',str(ASSETS/'sidecar_programs'),'--positive',POSITIVE,'--negative',NEGATIVE,'--queries',str(QUERIES),'--k',str(K),'--ef',str(EF),'--m',str(M),'--ef-construction',str(EF_CONSTRUCTION),'--gate-logprob',str(gate),'--postfilter-oversample',str(POSTFILTER_OVERSAMPLE),'--out',str(out)]
    print(f'\n=== {frac:.3f} eligible, gate={gate:.6f} ===')
    t0=time.time(); run=subprocess.run(cmd,text=True,capture_output=True)
    print(run.stdout)
    if run.returncode != 0:
        print(run.stderr); raise RuntimeError(f'fair live run failed for {frac}')
    z=pd.read_csv(out); z['target_fraction']=frac; z['gate_logprob']=gate; all_runs.append(z)
    print('wall seconds:',round(time.time()-t0,2))
results=pd.concat(all_runs,ignore_index=True)
print('rows:',len(results))


In [ ]:
#@title 8) Paper-grade summary
import numpy as np, pandas as pd
summary=(results.groupby(['target_fraction','method']).agg(queries=('query_id','count'),mean_latency_ms=('latency_ms','mean'),p50_latency_ms=('latency_ms','median'),p95_latency_ms=('latency_ms',lambda x:np.quantile(x,.95)),mean_recall_at_k=('recall_at_k','mean'),mean_returned=('returned','mean'),mean_visited=('visited','mean'),mean_semantic_evals=('semantic_evals','mean'),mean_predicate_evals=('predicate_evals','mean'),mean_dense_pruned_before_semantic=('dense_pruned_before_semantic','mean'),qualified_fraction=('qualified_fraction','mean'),live_match_rate=('live_matches_materialized','mean')).reset_index())
display(summary.sort_values(['target_fraction','mean_latency_ms'],ascending=[False,True]))
piv_t=summary.pivot(index='target_fraction',columns='method',values='mean_latency_ms')
piv_r=summary.pivot(index='target_fraction',columns='method',values='mean_recall_at_k')
rows=[]
for f in sorted(summary.target_fraction.unique(), reverse=True):
    live=summary[(summary.target_fraction==f)&(summary.method=='semantic_hnsw_live')].iloc[0]
    filt=summary[(summary.target_fraction==f)&(summary.method=='hnsw_filtered_materialized')].iloc[0]
    mat=summary[(summary.target_fraction==f)&(summary.method=='custom_hnsw_materialized')].iloc[0]
    overhead=float(live.mean_latency_ms-mat.mean_latency_ms)
    rows.append({'target_fraction':f,'live_latency_ms':float(live.mean_latency_ms),'filtered_same_dot_ms':float(filt.mean_latency_ms),'live_over_filtered_same_dot':float(live.mean_latency_ms/filt.mean_latency_ms),'live_minus_filtered_recall':float(live.mean_recall_at_k-filt.mean_recall_at_k),'live_program_overhead_ms':overhead,'predicate_evals':float(live.mean_predicate_evals),'approx_ns_per_predicate_eval':float(overhead*1e6/max(1,live.mean_predicate_evals)),'exact_id_parity':float(live.live_match_rate)})
fair=pd.DataFrame(rows)
display(fair)
print('This table is the fairness result: both HNSW baseline and custom traversal use the same normalized-dot geometry.')


In [ ]:
#@title 9) Plot fair recall-latency frontier
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(8,6))
for method,g in summary.groupby('method'):
    ax.plot(g.mean_latency_ms,g.mean_recall_at_k,marker='o',label=method)
    for _,r in g.iterrows(): ax.annotate(f'{r.target_fraction:.0%}',(r.mean_latency_ms,r.mean_recall_at_k),xytext=(4,4),textcoords='offset points',fontsize=8)
ax.set_xlabel('Mean query latency (ms)'); ax.set_ylabel(f'Mean Recall@{K}'); ax.set_title('Fair normalized-dot semantic HNSW frontier'); ax.legend(); ax.grid(True,alpha=.25); plt.show()


In [ ]:
#@title 10) Package results
import pathlib, shutil, json, platform, subprocess
PKG=pathlib.Path('/content/semantic_hnsw_fair_artifact')
if PKG.exists(): shutil.rmtree(PKG)
PKG.mkdir()
summary.to_csv(PKG/'summary.csv',index=False); fair.to_csv(PKG/'fair_comparison.csv',index=False); gate_df.to_csv(PKG/'gates.csv',index=False)
meta={'commit':subprocess.check_output(['git','-C','/content/ras','rev-parse','HEAD']).decode().strip(),'full_data':FULL_DATA,'indexed_items':int(len(items)),'queries':QUERIES,'k':K,'ef':EF,'m':M,'positive':POSITIVE,'negative':NEGATIVE,'distance':'1-clamp(dot,-1,1) on pre-normalized embeddings for both hnsw_rs baseline and custom traversal','cpu':next((l.split(':',1)[1].strip() for l in pathlib.Path('/proc/cpuinfo').read_text().splitlines() if l.startswith('model name')),'unknown'),'python':platform.python_version()}
(PKG/'environment.json').write_text(json.dumps(meta,indent=2))
shutil.make_archive('/content/rsa_semantic_hnsw_fair_full','zip',PKG)
print('/content/rsa_semantic_hnsw_fair_full.zip')
